In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")


In [4]:
from pinecone import Pinecone, ServerlessSpec

In [5]:
pc = Pinecone(api_key=pinecone_api_key)

In [6]:
pc.list_indexes()

IndexList([<name='my-index', dim=3, ready=True>])

In [6]:
idx = pc.Index(
    name="my-index")

In [8]:
emails_with_subject = 40
emails_with_body = 30
emails_with_other = 50

In [7]:
import numpy as np

In [15]:
np.random.rand(5, 3) # 5 rows with 3 columns 
vector_sub = np.random.rand(emails_with_subject, 3).tolist() # 40 rows with 3 columns
vector_body = np.random.rand(emails_with_body, 3).tolist() # 30 rows with 3 columns
vector_other = np.random.rand(emails_with_other, 3).tolist() # 50 rows with 3 columns

In [21]:
# create ids for this vector
ids_subject = [str(x) for x in np.arange(emails_with_subject).tolist()] # 0 to 39
ids_body = [str(x) for x in np.arange(emails_with_body).tolist()] # 0 to 29
ids_other = [str(x) for x in np.arange(emails_with_other).tolist()] # 0 to 49

In [22]:
# zip id and vector together
vectors_subject = list(zip(ids_subject, vector_sub))
vectors_body = list(zip(ids_body, vector_body))
vectors_other = list(zip(ids_other, vector_other))

In [24]:
# upsert vectors to the index with different namespaces
idx.upsert(vectors=vectors_subject, namespace="subject")
idx.upsert(vectors=vectors_body, namespace="body")
idx.upsert(vectors=vectors_other, namespace="other")    

KeyboardInterrupt: 

In [8]:
data = [
    {"id": "vec1", "values": [0.1, 0.2, 0.3], "metadata": {"category": "A"}},
    {"id": "vec2", "values": [0.4, 0.5, 0.6], "metadata": {"category": "B"}},
    {"id": "vec3", "values": [0.7, 0.8, 0.9], "metadata": {"category": "C"}} # only floats allowed
]

idx.upsert(vectors=data)

UpsertResponse(upserted_count=3)

In [11]:
# QUERY

idx.query(
    vector=[0.1, 0.2, 0.3],
    top_k=2,
    include_metadata=True,
    include_values=True,
    filter={"category": {"$eq": "A"}}
)

QueryResponse(matches=[ScoredVector(id='vec1', score=0.998880804, values=[0.1, 0.2, 0.3], metadata={'category': 'A'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 30 Jun 2026 09:02:25 GMT', 'content-type': 'application/json', 'content-length': '150', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '4', 'x-pinecone-request-latency-ms': '10', 'x-envoy-upstream-service-time': '10', 'x-pinecone-response-duration-ms': '12', 'grpc-status': '0', 'server': 'envoy'}))

In [13]:
idx.update(id="vec1", values=[0.15, 0.25, 0.35], set_metadata={"category": "D", "updated": True})

UpdateResponse(matched_records=None, response_info=ResponseInfo(raw_headers={'date': 'Tue, 30 Jun 2026 09:04:17 GMT', 'content-type': 'application/json', 'content-length': '2', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '5', 'x-pinecone-request-logical-size': '48', 'x-pinecone-request-latency-ms': '162', 'x-envoy-upstream-service-time': '162', 'x-pinecone-response-duration-ms': '164', 'grpc-status': '0', 'server': 'envoy'}))